# Tugas Laboratorium JS06 — Support Vector Machine

Eksperimen menggunakan `random_state=42` dan split terstratifikasi agar proporsi kelas terjaga. Akurasi dihitung sebagai jumlah prediksi benar dibagi jumlah data uji. Semua scaler hanya dipelajari dari data latih melalui pipeline.

## 1. Klasifikasi suara
Membandingkan kernel linear, polynomial (derajat 3), dan RBF pada rasio latih:uji 70:30 serta 80:20. Parameter dasar: `C=1`, `gamma="scale"`, dan `coef0=0`.

In [1]:
from pathlib import Path
import hashlib
import cv2
import numpy as np
import pandas as pd
from IPython.display import display, Markdown
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

SEED = 42
DATA_DIR = Path('dataset')
voice = pd.read_csv(DATA_DIR / 'voice.csv')
print('Dimensi data suara:', voice.shape)
print('Jumlah nilai kosong:', int(voice.isna().sum().sum()))
display(voice['label'].value_counts().rename_axis('Label').to_frame('Jumlah'))
display(voice.head())

Dimensi data suara: (3168, 21)
Jumlah nilai kosong: 0


,Jumlah
Label,
male,1584
female,1584


,meanfreq,sd,median,Q25,Q75,IQR,skew,kurt,sp.ent,sfm,mode,centroid,meanfun,minfun,maxfun,meandom,mindom,maxdom,dfrange,modindx,label
0,0.059781,0.064241,0.032027,0.015071,0.090193,0.075122,12.863462,274.402906,0.893369,0.491918,0.000000,0.059781,0.084279,0.015702,0.275862,0.007812,0.007812,0.007812,0.000000,0.000000,male
1,0.066009,0.067310,0.040229,0.019414,0.092666,0.073252,22.423285,634.613855,0.892193,0.513724,0.000000,0.066009,0.107937,0.015826,0.250000,0.009014,0.007812,0.054688,0.046875,0.052632,male
2,0.077316,0.083829,0.036718,0.008701,0.131908,0.123207,30.757155,1024.927705,0.846389,0.478905,0.000000,0.077316,0.098706,0.015656,0.271186,0.007990,0.007812,0.015625,0.007812,0.046512,male
3,0.151228,0.072111,0.158011,0.096582,0.207955,0.111374,1.232831,4.177296,0.963322,0.727232,0.083878,0.151228,0.088965,0.017798,0.250000,0.201497,0.007812,0.562500,0.554688,0.247119,male
4,0.135120,0.079146,0.124656,0.078720,0.206045,0.127325,1.101174,4.333713,0.971955,0.783568,0.104261,0.135120,0.106398,0.016931,0.266667,0.712812,0.007812,5.484375,5.476562,0.208274,male


In [2]:
X_voice = voice.drop(columns='label')
y_voice = voice['label']
assert X_voice.select_dtypes(include=np.number).shape[1] == X_voice.shape[1]
assert not voice.isna().any().any()

hasil_voice = []
model_voice = {}
for test_size, rasio in [(0.30, '70:30'), (0.20, '80:20')]:
    X_train, X_test, y_train, y_test = train_test_split(
        X_voice, y_voice, test_size=test_size, stratify=y_voice, random_state=SEED
    )
    for kernel in ['linear', 'poly', 'rbf']:
        model = Pipeline([
            ('scaler', StandardScaler()),
            ('svc', SVC(kernel=kernel, C=1.0, degree=3, gamma='scale', coef0=0.0))
        ])
        model.fit(X_train, y_train)
        prediksi = model.predict(X_test)
        model_voice[(rasio, kernel)] = model
        hasil_voice.append({
            'Rasio latih:uji': rasio, 'Kernel': kernel,
            'Jumlah latih': len(X_train), 'Jumlah uji': len(X_test),
            'Akurasi uji (%)': accuracy_score(y_test, prediksi) * 100
        })

tabel_voice = pd.DataFrame(hasil_voice)
display(tabel_voice.round(2))
display(tabel_voice.pivot(index='Kernel', columns='Rasio latih:uji', values='Akurasi uji (%)').round(2))

,Rasio latih:uji,Kernel,Jumlah latih,Jumlah uji,Akurasi uji (%)
0,70:30,linear,2217,951,97.90
1,70:30,poly,2217,951,96.00
2,70:30,rbf,2217,951,98.32
3,80:20,linear,2534,634,97.48
4,80:20,poly,2534,634,95.58
5,80:20,rbf,2534,634,98.26


Rasio latih:uji,70:30,80:20
Kernel,,
linear,97.90,97.48
poly,96.00,95.58
rbf,98.32,98.26


In [3]:
terbaik_voice = tabel_voice.loc[tabel_voice['Akurasi uji (%)'].idxmax()]
display(Markdown(
    f"Akurasi uji tertinggi pada eksperimen suara adalah **{terbaik_voice['Akurasi uji (%)']:.2f}%**, "
    f"dengan kernel **{terbaik_voice['Kernel']}** dan split **{terbaik_voice['Rasio latih:uji']}**. "
    'Perbandingan kernel dalam satu rasio menggunakan data latih dan uji yang sama. '
    'Hasil ini berlaku pada pembagian data tersebut; satu split belum membuktikan keunggulan universal suatu kernel.'
))

Akurasi uji tertinggi pada eksperimen suara adalah **98.32%**, dengan kernel **rbf** dan split **70:30**. Perbandingan kernel dalam satu rasio menggunakan data latih dan uji yang sama. Hasil ini berlaku pada pembagian data tersebut; satu split belum membuktikan keunggulan universal suatu kernel.

## 2. Klasifikasi siang dan malam dengan histogram dan SVM RBF

In [4]:
image_records = []
seen = {}
duplikat = 0
for folder in ['training', 'test']:
    for label in ['day', 'night']:
        direktori = DATA_DIR / 'images' / folder / label
        if not direktori.is_dir():
            raise FileNotFoundError(direktori)
        for path in sorted(direktori.glob('*.jpg')):
            image = cv2.imread(str(path))
            if image is None:
                raise ValueError(f'Gambar gagal dibaca: {path}')
            digest = hashlib.sha256(str(image.shape).encode() + image.tobytes()).hexdigest()
            if digest in seen:
                if seen[digest] != label:
                    raise ValueError(f'Gambar identik memiliki label berbeda: {path}')
                duplikat += 1
                continue
            seen[digest] = label
            image_records.append({'path': path, 'label': label, 'folder_asal': folder})

images = pd.DataFrame(image_records)
print('Duplikat gambar yang dihapus:', duplikat)
print('Jumlah gambar unik:', len(images))
display(pd.crosstab(images['folder_asal'], images['label']))

def histogram_hsv(path, bins=32):
    image = cv2.imread(str(path))
    if image is None:
        raise ValueError(f'Gambar gagal dibaca: {path}')
    image = cv2.resize(image, (128, 128), interpolation=cv2.INTER_AREA)
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    fitur = []
    for channel, batas in [(0, 180), (1, 256), (2, 256)]:
        hist = cv2.calcHist([hsv], [channel], None, [bins], [0, batas]).ravel()
        fitur.append(hist / hist.sum())
    return np.concatenate(fitur)

X_image = np.vstack([histogram_hsv(path) for path in images['path']])
y_image = images['label'].to_numpy()
idx_train, idx_test = train_test_split(
    np.arange(len(images)), test_size=0.20, stratify=y_image, random_state=SEED
)
X_img_train, X_img_test = X_image[idx_train], X_image[idx_test]
y_img_train, y_img_test = y_image[idx_train], y_image[idx_test]
assert set(idx_train).isdisjoint(idx_test)
print('Dimensi fitur:', X_image.shape)
display(pd.DataFrame({
    'Latih': pd.Series(y_img_train).value_counts(),
    'Uji': pd.Series(y_img_test).value_counts()
}).rename_axis('Label'))

Duplikat gambar yang dihapus: 1
Jumlah gambar unik: 399
Dimensi fitur: (399, 96)


label,day,night
folder_asal,,
test,79,80
training,120,120


,Latih,Uji
Label,,
night,160,40
day,159,40


In [5]:
baseline = Pipeline([
    ('scaler', StandardScaler()),
    ('svc', SVC(kernel='rbf', C=1.0, gamma='scale'))
])
baseline.fit(X_img_train, y_img_train)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
grid = GridSearchCV(
    estimator=Pipeline([('scaler', StandardScaler()), ('svc', SVC(kernel='rbf'))]),
    param_grid={
        'svc__C': [0.1, 1, 10, 100],
        'svc__gamma': ['scale', 0.001, 0.01, 0.1, 1]
    },
    scoring='accuracy', cv=cv, n_jobs=1, refit=True, error_score='raise'
)
grid.fit(X_img_train, y_img_train)
print('Parameter terbaik:', grid.best_params_)
print(f'Akurasi rata-rata validasi silang terbaik: {grid.best_score_ * 100:.2f}%')
hasil_cv = pd.DataFrame(grid.cv_results_)
display(hasil_cv[[
    'param_svc__C', 'param_svc__gamma', 'mean_test_score', 'std_test_score', 'rank_test_score'
]].sort_values('rank_test_score').head(10).reset_index(drop=True))

Parameter terbaik: {'svc__C': 10, 'svc__gamma': 0.001}
Akurasi rata-rata validasi silang terbaik: 99.69%


,param_svc__C,param_svc__gamma,mean_test_score,std_test_score,rank_test_score
0,10.0,0.001,0.996875,0.006250,1
1,100.0,0.001,0.996825,0.006349,2
2,1.0,scale,0.990526,0.012674,3
3,1.0,0.001,0.990526,0.012674,3
4,10.0,0.01,0.990526,0.012674,3
5,100.0,scale,0.990526,0.012674,3
6,10.0,scale,0.990526,0.012674,3
7,1.0,0.01,0.990526,0.012674,3
8,100.0,0.01,0.990526,0.012674,3
9,100.0,0.1,0.943700,0.037706,10


In [6]:
hasil_image = []
for nama, model in [('RBF dasar', baseline), ('RBF hasil tuning', grid.best_estimator_)]:
    pred = model.predict(X_img_test)
    hasil_image.append({
        'Model': nama, 'Rasio latih:uji': '80:20',
        'Jumlah latih': len(y_img_train), 'Jumlah uji': len(y_img_test),
        'C': model.named_steps['svc'].C, 'gamma': model.named_steps['svc'].gamma,
        'Akurasi uji (%)': accuracy_score(y_img_test, pred) * 100
    })
tabel_image = pd.DataFrame(hasil_image)
display(tabel_image.round(2))
pred_tuning = grid.predict(X_img_test)
display(pd.DataFrame(
    confusion_matrix(y_img_test, pred_tuning, labels=['day', 'night']),
    index=['Aktual siang', 'Aktual malam'], columns=['Prediksi siang', 'Prediksi malam']
))
print(classification_report(y_img_test, pred_tuning, labels=['day', 'night'],
                            target_names=['siang', 'malam'], digits=4))
display(Markdown(
    f"Model RBF dasar memperoleh akurasi uji **{hasil_image[0]['Akurasi uji (%)']:.2f}%**. "
    f"Setelah tuning, akurasi uji menjadi **{hasil_image[1]['Akurasi uji (%)']:.2f}%** "
    f"dengan `C={grid.best_params_['svc__C']}` dan `gamma={grid.best_params_['svc__gamma']}`. "
    'Parameter dipilih berdasarkan akurasi validasi silang, bukan akurasi data uji. '
    'Gambar yang mirip dari rangkaian waktu atau lokasi sama masih dapat muncul di kedua bagian; '
    'pengujian pada lokasi atau waktu lain diperlukan untuk menilai generalisasi yang lebih luas.'
))

              precision    recall  f1-score   support

       siang     1.0000    1.0000    1.0000        40
       malam     1.0000    1.0000    1.0000        40

    accuracy                         1.0000        80
   macro avg     1.0000    1.0000    1.0000        80
weighted avg     1.0000    1.0000    1.0000        80



,Model,Rasio latih:uji,Jumlah latih,Jumlah uji,C,gamma,Akurasi uji (%)
0,RBF dasar,80:20,319,80,1.0,scale,100.0
1,RBF hasil tuning,80:20,319,80,10.0,0.001,100.0


,Prediksi siang,Prediksi malam
Aktual siang,40,0
Aktual malam,0,40


Model RBF dasar memperoleh akurasi uji **100.00%**. Setelah tuning, akurasi uji menjadi **100.00%** dengan `C=10` dan `gamma=0.001`. Parameter dipilih berdasarkan akurasi validasi silang, bukan akurasi data uji. Gambar yang mirip dari rangkaian waktu atau lokasi sama masih dapat muncul di kedua bagian; pengujian pada lokasi atau waktu lain diperlukan untuk menilai generalisasi yang lebih luas.

## Ringkasan Akurasi Hasil Eksekusi

Akurasi berikut dihitung pada data uji dengan `random_state=42`. Rasio menunjukkan data latih:data uji.

### Klasifikasi suara ? `dataset/voice.csv`

| Kernel SVM | Akurasi split 70:30 | Akurasi split 80:20 |
|---|---:|---:|
| Linear | 97,90% | 97,48% |
| Polynomial (derajat 3) | 96,00% | 95,58% |
| RBF | 98,32% | 98,26% |

Akurasi tertinggi pada eksperimen suara adalah **98,32%**, menggunakan kernel **RBF** dengan split **70:30**.

### Klasifikasi siang dan malam ? histogram HSV, split 80:20

| Model | C | gamma | Jumlah data latih | Jumlah data uji | Akurasi uji |
|---|---:|---|---:|---:|---:|
| SVM RBF dasar | 1 | scale | 319 | 80 | 100,00% |
| SVM RBF hasil tuning | 10 | 0.001 | 319 | 80 | 100,00% |

Kedua model memprediksi **80 dari 80 gambar uji dengan benar**. Tuning menggunakan validasi silang 5-fold pada data latih; akurasi uji tetap **100,00%**. Hasil ini berlaku pada split yang digunakan dan belum menjamin akurasi yang sama pada gambar dari lokasi atau waktu lain.
